# Visualization
Paper-quality figures for the robust CFLP study.  
Edit the **Configuration** cell, then run the figure cells you need.

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
# Path to the Excel file produced by 05_robustification_and_adaptive_pricing.ipynb
EXCEL = "05_sensitivity_results.xlsx"   # relative to experiments/, or use an absolute path

# Default slice for the violin plot
V_SCALE = 0.75
W_VAL   = 10
GAMMAS  = [1, 2, 3, 4]

# Full parameter grid (used by multi-panel figures)
V_LIST = [0.75, 1.0, 1.25]
W_LIST = [10, 100, 1000]

In [ ]:
import importlib, sys, pathlib

# Make sure experiments/ is on the path so we can import fig_violin_cvar
_here = pathlib.Path().resolve()
if str(_here) not in sys.path:
    sys.path.insert(0, str(_here))

import fig_violin_cvar as fvc
importlib.reload(fvc)   # picks up any edits without restarting the kernel

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
import matplotlib.lines as mlines

%matplotlib inline
plt.rcParams.update(fvc.mpl.rcParams)   # reuse the same style

In [ ]:
# ── Load data ─────────────────────────────────────────────────────────────────
oos  = pd.read_excel(EXCEL, sheet_name="OOS_Raw")
sens = pd.read_excel(EXCEL, sheet_name="Summary")
print(f"OOS rows : {len(oos)}  |  columns: {list(oos.columns)}")
print(f"Summary  : {len(sens)} rows  |  columns: {list(sens.columns)}")

---
## Fig 1 — Violin + CVaR 5%: Nominal vs Robust (OOS profit distribution)
One pair of violins per Γ. Dark tail = bottom-5% of the distribution; bold bar = CVaR₅%.

In [ ]:
HALF_W  = 0.32
GAP     = 0.08
GROUP_W = 2.0

sub = oos[(oos["v_scale"] == V_SCALE) & (oos["w"] == W_VAL)]

fig, ax = plt.subplots(figsize=(10, 5.5))
x_ticks, x_labels, cvar_summary = [], [], {}

for gi, gam in enumerate(GAMMAS):
    g = sub[sub["gamma"] == gam]
    arr_nom = g["profit_a_nom"].values
    arr_rob = g["profit_a_rob"].values

    x_base = gi * GROUP_W
    x_nom  = x_base - (HALF_W + GAP / 2)
    x_rob  = x_base + (HALF_W + GAP / 2)

    cv_nom = fvc.draw_violin(ax, arr_nom, x_nom, HALF_W, fvc.NOM_LIGHT, fvc.NOM_DARK,
                             label="Nominal $x$" if gi == 0 else None)
    cv_rob = fvc.draw_violin(ax, arr_rob, x_rob, HALF_W, fvc.ROB_LIGHT, fvc.ROB_DARK,
                             label="Robust $x$" if gi == 0 else None)

    cvar_summary[gam] = (cv_nom, cv_rob)
    x_ticks.append(x_base)
    x_labels.append(f"$\\Gamma = {gam}$")

ax.axhline(0, color="black", lw=0.9, linestyle=":", zorder=1, alpha=0.6)
ax.set_xticks(x_ticks)
ax.set_xticklabels(x_labels)
ax.set_xlim(x_ticks[0] - GROUP_W * 0.7, x_ticks[-1] + GROUP_W * 0.7)
ax.set_ylabel("Out-of-sample profit")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"${x/1e3:.0f}k"))
ax.grid(axis="y", linestyle="--")

legend_patches = [
    mpatches.Patch(color=fvc.NOM_LIGHT, alpha=0.7, label="Nominal $x$  (Scen. A)"),
    mpatches.Patch(color=fvc.ROB_LIGHT, alpha=0.7, label="Robust $x$   (Scen. A)"),
    mpatches.Patch(color=fvc.NOM_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Nominal"),
    mpatches.Patch(color=fvc.ROB_DARK,  alpha=0.85, label="CVaR$_{5\\%}$ tail — Robust"),
    plt.Line2D([0],[0], color="gray", lw=2.2, label="Median"),
    plt.Line2D([0],[0], color="gray", lw=1.2, linestyle="--", label="5th percentile (VaR)"),
    plt.Line2D([0],[0], marker="D", color="w", markeredgecolor="gray", markersize=5, label="Mean"),
]
ax.legend(handles=legend_patches, ncol=2, frameon=True, framealpha=0.92,
          loc="upper right", fontsize=8.5, edgecolor="0.8")
ax.set_title(
    f"OOS Profit Distributions — Nominal vs. Robust  "
    f"($v = {V_SCALE},\\; w = {W_VAL}$, adaptive pricing)",
    fontsize=11, pad=8)

fig.tight_layout()
fig.savefig("fig1_violin_cvar.pdf", bbox_inches="tight")
fig.savefig("fig1_violin_cvar.png", dpi=300, bbox_inches="tight")
plt.show()

print(f"\n{'Γ':>4}  {'CVaR5 Nom':>12}  {'CVaR5 Rob':>12}  {'Δ CVaR5':>12}")
print("-" * 46)
for gam, (cn, cr) in cvar_summary.items():
    print(f"{gam:>4}  {cn:>12,.0f}  {cr:>12,.0f}  {cr-cn:>+12,.0f}")

---
## Fig 2 — Price of Robustness (PoR) & Value of Robustification (VoR)

**PoR** measures the profit *sacrificed* by the robust design when no disruption occurs:  
$\text{PoR}(\%) = 100 \times (\pi^{\text{nom}}_0 - \pi^{\text{rob}}_0)\;/\;\pi^{\text{nom}}_0$

**VoR** measures the profit *gained* by the robust design in the worst-case disruption:  
$\text{VoR} = \pi^{\text{rob}}_{\text{wc}} - \pi^{\text{nom}}_{\text{wc}}$

Layout: 2 rows × 3 columns (one column per *w*).  
Top row = PoR (%), bottom row = VoR (absolute). Lines coloured by *v*.

In [ ]:
# ── colour / marker scheme for v values ───────────────────────────────────────
V_COLORS  = {0.75: "#1b7837", 1.0: "#762a83", 1.25: "#d6604d"}
V_MARKERS = {0.75: "o",        1.0: "s",        1.25: "^"}  
V_LABELS  = {0.75: "$v=0.75$", 1.0: "$v=1.00$", 1.25: "$v=1.25$"}

W_TITLES  = {10: "$w = 10$ (low congestion)",
             100: "$w = 100$ (moderate)",
             1000: "$w = 1000$ (high congestion)"}

fig, axes = plt.subplots(2, 3, figsize=(13, 7), sharey="row")

for col_idx, w in enumerate(W_LIST):
    ax_por = axes[0, col_idx]
    ax_vor = axes[1, col_idx]

    for v in V_LIST:
        sub = sens[(sens["v_scale"] == v) & (sens["w"] == w)].sort_values("gamma")
        gammas = sub["gamma"].values

        # PoR (%): how much profit the robust solution sacrifices under no disruption
        por = 100 * (sub["nd_nom"].values - sub["nd_rob"].values) / np.abs(sub["nd_nom"].values)

        # VoR: extra profit from robust solution under worst-case disruption
        vor = sub["wc_rob_val_a"].values

        kw = dict(color=V_COLORS[v], marker=V_MARKERS[v],
                  markersize=6, linewidth=1.6, label=V_LABELS[v])

        ax_por.plot(gammas, por, **kw)
        ax_vor.plot(gammas, vor, **kw)

    # PoR row formatting
    ax_por.axhline(0, color="black", lw=0.7, linestyle=":", alpha=0.5)
    ax_por.set_title(W_TITLES[w], fontsize=10)
    ax_por.set_xticks(GAMMAS)
    ax_por.set_xticklabels([f"$\\Gamma={g}$" for g in GAMMAS])
    ax_por.yaxis.set_major_formatter(mticker.FuncFormatter(lambda y, _: f"{y:.1f}%"))
    ax_por.grid(axis="y", linestyle="--", alpha=0.4)
    if col_idx == 0:
        ax_por.set_ylabel("Price of Robustness PoR (%)", fontsize=10)

    # VoR row formatting
    ax_vor.axhline(0, color="black", lw=0.7, linestyle=":", alpha=0.5)
    ax_vor.set_xticks(GAMMAS)
    ax_vor.set_xticklabels([f"$\\Gamma={g}$" for g in GAMMAS])
    ax_vor.yaxis.set_major_formatter(
        mticker.FuncFormatter(lambda y, _: f"${y/1e3:.0f}k"))
    ax_vor.grid(axis="y", linestyle="--", alpha=0.4)
    if col_idx == 0:
        ax_vor.set_ylabel("Value of Robustification VoR", fontsize=10)

# Shared legend (top-right panel)
handles = [
    mlines.Line2D([], [], color=V_COLORS[v], marker=V_MARKERS[v],
                  markersize=6, linewidth=1.6, label=V_LABELS[v])
    for v in V_LIST
]
axes[0, 2].legend(handles=handles, frameon=True, framealpha=0.9,
                   fontsize=9, loc="upper left")

fig.suptitle(
    "Cost vs. Benefit of Robustification — PoR (top) and VoR (bottom)",
    fontsize=12, y=1.01)

fig.tight_layout()
fig.savefig("fig2_por_vor.pdf", bbox_inches="tight")
fig.savefig("fig2_por_vor.png", dpi=300, bbox_inches="tight")
plt.show()

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'v':>5}  {'w':>5}  {'Γ':>3}  {'nd_nom':>10}  {'nd_rob':>10}  {'PoR%':>7}  {'VoR':>10}")
print("-" * 60)
for v in V_LIST:
    for w in W_LIST:
        sub = sens[(sens["v_scale"] == v) & (sens["w"] == w)].sort_values("gamma")
        for _, r in sub.iterrows():
            por_val = 100 * (r.nd_nom - r.nd_rob) / abs(r.nd_nom)
            print(f"{v:>5.2f}  {w:>5.0f}  {r.gamma:>3.0f}  "
                  f"{r.nd_nom:>10,.0f}  {r.nd_rob:>10,.0f}  "
                  f"{por_val:>7.2f}%  {r.wc_rob_val_a:>10,.0f}")